In [11]:
import os
from dotenv import load_dotenv

# scaper
from tavily import TavilyClient

# output class
from pydantic import BaseModel
from typing import Optional
from datetime import datetime

# langchain
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

# fixed output
from pydantic import BaseModel, Field
from typing import Optional, List
from langchain_core.output_parsers import PydanticOutputParser
#from langchain_core.output_parsers import JsonOutputParser

In [2]:
load_dotenv()

True

In [ ]:
tool = TavilySearchResults(
    max_results=5,
    search_depth="basic" # or advanced [to have 'raw_text' option]
)

llm = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
# schema for a single article
class NewsArticle(BaseModel):
    title: str
    summary: str = Field(description="2-4 sentence summary")
    url: str
    topic: str = Field(description="AI / Tech / Finance / Politics / Other")
    source: str = Field(description="domain only, e.g. bbc.com")
    timestamp: Optional[str] = Field(default=None, description="ISO 8601, publication date")
    raw_text: Optional[str] = Field(default=None, description="null if paywalled")
    language: str = Field(description="ISO 639-1, e.g. en, pl")
    fetch_score: Optional[float] = Field(default=None, description="Tavily relevance score 0-1")
    paywall: bool = Field(default=False)

class NewsFeed(BaseModel):
    articles: List[NewsArticle]

# parser = JsonOutputParser(pydantic_object=NewsFeed)
parser = PydanticOutputParser(pydantic_object=NewsFeed)

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a news fetching agent that retrieves articles from the internet.

Return results ONLY as JSON — no text before or after.
Do not add markdown, do not write ```json, just raw JSON.

Response format:
{format_instructions}

Rules:
- source: domain name only (e.g. bbc.com, techcrunch.com)
- timestamp: publication date in ISO 8601, NOT the fetch date. If unknown — null
- language: detect the article language (en, pl, de, etc.)
- fetch_score: use the score returned by Tavily (0.0 - 1.0)
- paywall: true if content is unavailable or truncated
- raw_text: full content from Tavily, null if paywalled
"""),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
]).partial(format_instructions=parser.get_format_instructions())

In [13]:
agent = create_tool_calling_agent(llm, [tool], prompt)
executor = AgentExecutor(agent=agent, tools=[tool], verbose=False)

In [ ]:
result = executor.invoke({
    "input": "Find 5 newest AI articles from today"
})

# parse to the structure
feed = parser.parse(result["output"])

In [25]:
feed.articles[0].title
feed.articles[0].summary
#feed.articles[0].fetch_score
feed.articles[1].raw_text

In [19]:
for article in feed.articles:
    print(f"[{article.topic}] {article.title}")
    print(f"  source: {article.source} | score: {article.fetch_score} | paywall: {article.paywall}")
    print(f"  {article.summary[:100]}...")
    print()

[AI] AI News: The Model That Has Everyone Freaked Out!
  source: jvfocus.com | score: 0.73846215 | paywall: False
  Leading AI expert Matt Wolfe discusses a groundbreaking AI model that has captured widespread attent...

[AI] Artificial Intelligence - Latest AI News and Analysis
  source: wsj.com | score: 0.9999821 | paywall: False
  The latest coverage of artificial intelligence developments, including tools and technologies shapin...

[AI] AI News | Latest News | Insights Powering AI-Driven Business Growth
  source: artificialintelligence-news.com | score: 0.99999297 | paywall: False
  Delivering the latest updates in artificial intelligence, machine learning, and their impact on glob...

[AI] Powering AI: How Growth Drives Surge in Power Demand
  source: morningstar.com | score: 0.9998313 | paywall: False
  Exploring how the rise of AI is contributing to increased energy demands and expanding data center c...

[AI] AI Is Powering Small Business Growth in 2026
  source: uschamber.com